In [25]:
# =========================
# INSTALAR LIBRERÍAS
# =========================

!pip install altair pandas vl-convert-python pillow -q

# =========================
# IMPORTAR LIBRERÍAS
# =========================

import pandas as pd
import altair as alt
from PIL import Image

# =========================
# CARGAR BASE DE DATOS
# =========================

df = pd.read_csv(
    "Base de datos Wikipedia.csv",
    sep=';'
)

# =========================
# LIMPIEZA DE DATOS
# =========================

df["Total Visitas (periodo seis meses)"] = pd.to_numeric(
    df["Total Visitas (periodo seis meses)"],
    errors="coerce"
)

df = df.dropna(
    subset=["Total Visitas (periodo seis meses)"]
)

# =========================
# ORDENAR DATOS
# =========================

df = df.sort_values(
    by="Total Visitas (periodo seis meses)",
    ascending=False
)

# =========================
# TOP 10 Y BOTTOM 10
# =========================

top10 = df.head(10)
bottom10 = df.tail(10)

# =========================
# ESPECIES MEDIÁTICAS
# =========================

mediaticas = df[
    df["Nombre Científico"].isin([
        "Hippocamelus bisulcus",
        "Rhinoderma darwinii",
        "Spheniscus humboldti"
    ])
]

# =========================
# UNIR BASES
# =========================

final_df = pd.concat([
    top10,
    bottom10,
    mediaticas
]).drop_duplicates()

# =========================
# CREAR ETIQUETA
# =========================

def crear_etiqueta(row):

    nombre_comun = str(row["Nombre Común"])

    if nombre_comun == "nan":
        primer_nombre = "Sin nombre común"

    else:
        primer_nombre = nombre_comun.split(",")[0]

    return f'{row["Nombre Científico"]} ({primer_nombre})'

final_df["Etiqueta"] = final_df.apply(
    crear_etiqueta,
    axis=1
)

# =========================
# CATEGORIZAR ESPECIES
# =========================

final_df["Grupo"] = "Menos visitadas"

# TOP 10
top10_etiquetas = top10.apply(
    crear_etiqueta,
    axis=1
)

final_df.loc[
    final_df["Etiqueta"].isin(top10_etiquetas),
    "Grupo"
] = "Más visitadas"

# ESPECIES MEDIÁTICAS
final_df.loc[
    final_df["Nombre Científico"].isin([
        "Hippocamelus bisulcus",
        "Rhinoderma darwinii",
        "Spheniscus humboldti"
    ]),
    "Grupo"
] = "Especies mediáticas"

# =========================
# ORDEN DEL GRÁFICO
# =========================

orden = final_df.sort_values(
    "Total Visitas (periodo seis meses)"
)["Etiqueta"].tolist()

# =========================
# CREAR GRÁFICO
# =========================

chart = alt.Chart(final_df).mark_circle(
    size=230
).encode(

    x=alt.X(
        "Total Visitas (periodo seis meses):Q",
        title="Total de visitas en Wikipedia (6 meses)"
    ),

    y=alt.Y(
        "Etiqueta:N",
        sort=orden,
        title="",
        axis=alt.Axis(
            labelLimit=500
        )
    ),

    color=alt.Color(
        "Grupo:N",

        scale=alt.Scale(
            domain=[
                "Más visitadas",
                "Menos visitadas",
                "Especies mediáticas"
            ],

            range=[
                "#2a9d8f",   # verde
                "#6c757d",   # gris
                "#d62828"    # rojo
            ]
        ),

        title=""
    ),

    tooltip=[

        alt.Tooltip(
            "Nombre Científico:N",
            title="Nombre científico"
        ),

        alt.Tooltip(
            "Nombre Común:N",
            title="Nombre común"
        ),

        alt.Tooltip(
            "Clase:N",
            title="Clase"
        ),

        alt.Tooltip(
            "RCE:N",
            title="Categoría"
        ),

        alt.Tooltip(
            "Total Visitas (periodo seis meses):Q",
            title="Visitas"
        )
    ]

).properties(

    width=900,
    height=700,

    title={
        "text": "Las especies amenazadas que sí vemos y las que ignoramos",

        "subtitle": [
            "Top 10, bottom 10 y especies mediáticas según visitas en Wikipedia"
        ]
    }

).configure_title(

    fontSize=22,
    subtitleFontSize=14,
    anchor="start"

).configure_axis(

    labelFontSize=11,
    titleFontSize=13

).configure_legend(

    titleFontSize=13,
    labelFontSize=12

).configure_view(
    stroke=None
)

# =========================
# MOSTRAR GRÁFICO
# =========================

chart

# =========================
# EXPORTAR HTML
# =========================

chart.save("visualizacion.html")

# =========================
# EXPORTAR PNG
# =========================

chart.save("visualizacion.png")

# =========================
# CONVERTIR PNG A JPG
# =========================

img = Image.open("visualizacion.png")

rgb = img.convert("RGB")

rgb.save("visualizacion.jpg")

# =========================
# MENSAJE FINAL
# =========================

print("Visualización exportada correctamente")

Visualización exportada correctamente
